You need to do the followig before you run the code
- Change the user agent variable in the following code block.
- If you want to create a sub-index using the LLM, you need an Anthropic API key (sector/seniority classification runs on Claude).
- The output paths need to be defined.
- A directory will be created when you run section 1 of this code containing the data. You need to use data/processed/ixbrl_cleaned.csv
### You will need to defined the files such as cdli_csv_path, rates file, fx_path containing the relevant returns.
### Final Data.zip contains these files for reference of their structure.

In [ ]:
user_agent = "Your_Name your.email@domain.com"
API_KEY = None  # Anthropic API key; None => read from env var ANTHROPIC_API_KEY
OUTPUT_PATH_LLM = "data_private_credit.csv"
OUTPUT_PATH = "ixbrl_cleaned_out.csv"
cols = ["cik","investment_identifier","cal_q","FV", "COST", "PAR", "rate_cash", "interest_rate", "rate_pik"]
cols_llm = ["cik","investment_identifier","cal_q","FV", "COST", "PAR", "rate_cash", "interest_rate", "rate_pik","sector","instrument_seniority"]
interest_cols = ["rate_cash", "interest_rate", "rate_pik"]
cdli_csv_path="cdli.csv"
cdli_s_csv_path= "cdli-s.csv"
rates_file = "Search.xlsx"
data_path = "ixbrl_clean.csv"
fx_path = "FX.csv"
sofr_path = "SOFR.csv"
flag_llm = False


In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve().parents[0]  # 2026 Extension/
sys.path.append(str(ROOT / "src"))
sys.path.append(str(ROOT / "scripts"))


In [ ]:
from __future__ import annotations
import sys
from pathlib import Path
from run_bdc_universe import run_bdc_universe
from run_downloader import run_download
from run_ixbrl_parser import run_ixbrl_parse
from run_preprocessor import main as run_preprocess_main
from index_construction import *
from run_ixbrl_pipeline import *
from get_seniority_debttype import *
from subindex_construction import *


## Download the Data using SEC Edgar API
### Refer to README_1.md

In [ ]:
# 1) universe
run_bdc_universe(user_agent=user_agent)

In [ ]:
# 2) download full
run_download(user_agent=user_agent, mode="full")

In [ ]:
# 3) parse full (incremental)
run_ixbrl_parse(user_agent=user_agent, append_if_exists=True, resume_parsed_only=True)

In [ ]:
# 4) preprocess
run_preprocess_main()

## Cleaning the data
### Refer to README_2.MD

In [ ]:
df_clean = run_pipeline(
        data_path=data_path,
        fx_path=fx_path,
        sofr_path=sofr_path,
    )
df_clean.to_csv(OUTPUT_PATH, index=False)

## Adding the Sector and Seniority using LLM (Claude)
### Refer to README_3.MD
#### You do not need to run this section if you are trying to construct the main index. 
#### You only need this for the Senior Loans/ Sector Index

In [ ]:
flag_llm = True
build_final_dataset_with_seniority_and_sector(
        input_csv_path=OUTPUT_PATH,
        output_csv_path=OUTPUT_PATH_LLM,
        conf_thresh=0.70,
        llm_model="claude-haiku-4-5",
        batch_size=32,
        anthropic_api_key=API_KEY,  # None => uses env var ANTHROPIC_API_KEY
        verbose=True,
    )

## Index Construction Process
### Refer to README_4.MD

In [ ]:
if flag_llm:
    df = load_and_prepare_investment_data(
        OUTPUT_PATH_LLM
    )
    df = df.select(cols_llm)
else:
    df = load_and_prepare_investment_data(
        OUTPUT_PATH
    )
    df = df.select(cols)
df.head()

In [ ]:
missing_summary_pd = missing_percentage_summary(df, cols)
missing_summary_pd

In [ ]:
rate_combinations_pd = interest_rate_null_combinations(df, interest_cols)
rate_combinations_pd

In [ ]:
df, subindex_prep_report = prepare_subindex_input(df)
print("Rate clipping / sector-label merge report (see docs/README_5.md):")
subindex_prep_report


In [ ]:
df = compute_final_interest_rates(df)
df.head()

In [ ]:
df = compute_position_level_flows(
    df,
    qsort_expr=qsort_expr,
    safe_div=safe_div
)
df.head()

In [ ]:
quarter_distribution = quarter_count_distribution(df)
quarter_distribution

In [ ]:
quarterly_summary = quarterly_cik_entry_exit_aum(df)
quarterly_summary

In [ ]:
df = compute_market_weights(df, safe_div=safe_div)
df.head()

In [ ]:
df_decomp = aggregate_return_decomposition(df, safe_div)
df_decomp_pd = df_decomp.to_pandas()

plot_quarterly_return_decomposition(
    df_decomp_pd,
    title="Market-Level BDC Index – Quarterly Return Decomposition\n"
)

## Get Return Decomposition for a particular Sector
#### No need to run this for index construction of the main index

In [ ]:
tech_result = build_subindex(df, "sector", "Technology", ["Technology"])
df_decomp_pd = tech_result["decomp"].to_pandas()

plot_quarterly_return_decomposition(
    df_decomp_pd,
    title=f"Market-Level BDC Index – Quarterly Return Decomposition - Technology only\n"
)


## Get Return Decomposition for SENIOR LOANS
#### No need to run this for index construction of the main index

In [ ]:
loans_result = build_subindex(
    df, "instrument_seniority", "Senior Loans", DEFAULT_SENIOR_LOANS_BUCKET
)
df_loans = loans_result["positions"]
df_decomp_pd = loans_result["decomp"].to_pandas()
plot_quarterly_return_decomposition(
    df_decomp_pd,
    title=f"Market-Level BDC Index – Quarterly Return Decomposition - SENIOR Loans only\n"
)


## Full Sector and Seniority Sub-Index Sweeps
#### Generalizes the two ad hoc cells above via subindex_construction.py -- see docs/README_5.md


In [ ]:
sector_results = build_all_subindices_for_dimension(df, "sector")
sector_suppressed = summarize_suppressed_quarters(sector_results)
sector_suppressed


In [ ]:
seniority_results = build_all_subindices_for_dimension(df, "instrument_seniority")
seniority_suppressed = summarize_suppressed_quarters(seniority_results)
seniority_suppressed


In [ ]:
index_mkt_flow = compute_flow_based_market_index(df)
index_mkt_flow_pd = index_mkt_flow.to_pandas()
index_mkt_flow_pd

In [ ]:
plot_index_vs_cdli_returns(
    index_df=index_mkt_flow,
    cdli_csv_path=cdli_csv_path
)

## Plot CDLI-S with Index Senior Loans
#### No need to run this for index construction of the main index

In [ ]:
index_mkt_flow_loans = compute_flow_based_market_index(df_loans)
plot_index_vs_cdli_returns(
    index_df=index_mkt_flow_loans,
    cdli_csv_path=cdli_s_csv_path
)

In [ ]:
annual_return, vol_a = compute_annualized_return_and_vol(
    returns=index_mkt_flow["IndexReturn"],
    periods_per_year=4,
    start_idx=1,  # drop 2023Q1: no prior-quarter FV, so its return is undefined
)
annual_return, vol_a

In [ ]:
quarterly_rates = load_quarterly_rates(rates_file)
quarterly_rates

In [ ]:
plot_df = build_quarterly_comparison_df(
    quarterly_rates,
    index_mkt_flow,
    cdli_csv_path
)
plot_df

In [ ]:
corr, te_q, te_a = compute_tracking_error(plot_df)
corr,te_a

In [ ]:
plot_index_cdli_sofr(plot_df)

## Correlation & Tracking Error - Senior Loans Index vs CDLI-S
#### No need to run this for index construction of the main index

In [ ]:
plot_df_loans = build_quarterly_comparison_df(
    quarterly_rates,
    index_mkt_flow_loans,
    cdli_s_csv_path
)
plot_df_loans

In [ ]:
corr_loans, te_q_loans, te_a_loans = compute_tracking_error(plot_df_loans)
corr_loans, te_a_loans

In [ ]:
plot_index_cdli_sofr(
    plot_df_loans,
    title="Senior Loans Index Returns Comparison with CDLI-S and SOFR Quarterly Averages"
)

In [ ]:
a = (
    df.group_by("cal_q")
      .agg(
          pl.col("FV_prev").null_count().alias("null_count"),
          (pl.col("FV_prev").null_count() / pl.len()).alias("null_proportion"),
          pl.col("cik").n_unique().alias("unique_ciks")
      )
)

a = a.to_pandas()
a